# Workshop 1 (2026) - dlt Homework Answers

This notebook runs a dlt pipeline against the workshop API and computes the three homework answers.

In [1]:
import json
import dlt
from dlt.sources.helpers.rest_client import RESTClient
from dlt.sources.helpers.rest_client.paginators import PageNumberPaginator

BASE_URL = "https://us-central1-dlthub-analytics.cloudfunctions.net"
ENDPOINT = "data_engineering_zoomcamp_api"
PIPELINE_NAME = "workshop_2026_01_taxi_pipeline"
DATASET_NAME = "workshop_2026_01_dlt"
RESOURCE_NAME = "rides"

In [7]:
@dlt.resource(name=RESOURCE_NAME, write_disposition="replace")
def nyc_taxi_rides():
    client = RESTClient(
        base_url=BASE_URL,
        paginator=PageNumberPaginator(base_page=1, total_path=None),
    )

    for page in client.paginate(ENDPOINT):
        yield page

pipeline = dlt.pipeline(
    pipeline_name=PIPELINE_NAME,
    destination="duckdb",
    dataset_name=DATASET_NAME,
)

load_info = pipeline.run(nyc_taxi_rides())
#print(load_info)

2026-03-02 16:49:29,799|[WARNING]|1794779|125059484741824|dlt|validate.py|verify_normalized_table:91|In schema `workshop_2026_01_taxi`: The following columns in table 'rides' did not receive any data during this load and therefore could not have their types inferred:
  - rate_code
  - mta_tax

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'rate_code': {'data_type': 'text'}})



In [3]:
with pipeline.sql_client() as client:
    cols = client.execute_sql(
        f"""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = '{DATASET_NAME}'
          AND table_name = '{RESOURCE_NAME}'
        ORDER BY ordinal_position
        """
    )

available = {row[0] for row in cols}
pickup_col = "trip_pickup_date_time" if "trip_pickup_date_time" in available else "tpep_pickup_datetime"
dropoff_col = "trip_dropoff_date_time" if "trip_dropoff_date_time" in available else "tpep_dropoff_datetime"
tip_col = "tip_amount" if "tip_amount" in available else "tip_amt"
payment_col = "payment_type"

with pipeline.sql_client() as client:
    q1 = client.execute_sql(
        f"""
        SELECT
            MIN(CAST({pickup_col} AS DATE)) AS min_pickup_date,
            MAX(CAST({dropoff_col} AS DATE)) AS max_dropoff_date
        FROM {RESOURCE_NAME}
        """
    )[0]

print("Columns used:", {"pickup": pickup_col, "dropoff": dropoff_col, "tip": tip_col, "payment": payment_col})
print("Q1 raw result:", q1)

Columns used: {'pickup': 'trip_pickup_date_time', 'dropoff': 'trip_dropoff_date_time', 'tip': 'tip_amt', 'payment': 'payment_type'}
Q1 raw result: (datetime.date(2009, 6, 1), datetime.date(2009, 7, 1))


In [4]:
with pipeline.sql_client() as client:
    total_rows, credit_rows = client.execute_sql(
        f"""
        SELECT
            COUNT(*) AS total_rows,
            SUM(
                CASE
                    WHEN CAST({payment_col} AS VARCHAR) = '1' THEN 1
                    WHEN LOWER(CAST({payment_col} AS VARCHAR)) IN ('credit', 'credit card', 'card') THEN 1
                    ELSE 0
                END
            ) AS credit_rows
        FROM {RESOURCE_NAME}
        """
    )[0]

credit_pct = round((credit_rows * 100.0) / total_rows, 2) if total_rows else 0.0
print("Q2 raw result:", {"credit_rows": int(credit_rows), "total_rows": int(total_rows), "credit_pct": credit_pct})

Q2 raw result: {'credit_rows': 2666, 'total_rows': 10000, 'credit_pct': 26.66}


In [5]:
with pipeline.sql_client() as client:
    total_tips = client.execute_sql(
        f"""
        SELECT ROUND(SUM(COALESCE(CAST({tip_col} AS DOUBLE), 0.0)), 2)
        FROM {RESOURCE_NAME}
        """
    )[0][0]

print("Q3 raw result:", float(total_tips))

Q3 raw result: 6063.41


In [6]:
answers = {
    "q1": f"{q1[0]} to {q1[1]}",
    "q2": f"{credit_pct:.2f}%",
    "q3": f"${float(total_tips):,.2f}",
}

print(json.dumps(answers, indent=2))

q1_options = [
    "2009-01-01 to 2009-01-31",
    "2009-06-01 to 2009-07-01",
    "2024-01-01 to 2024-02-01",
    "2024-06-01 to 2024-07-01",
]
q2_options = ["16.66%", "26.66%", "36.66%", "46.66%"]
q3_options = ["$4,063.41", "$6,063.41", "$8,063.41", "$10,063.41"]

def pick_option(value: str, options: list[str]) -> str:
    if value in options:
        return value
    return f"No exact option match (computed={value})"

print("\nSelected options from execution:")
print(f"Q1 -> {pick_option(answers['q1'], q1_options)}")
print(f"Q2 -> {pick_option(answers['q2'], q2_options)}")
print(f"Q3 -> {pick_option(answers['q3'], q3_options)}")


{
  "q1": "2009-06-01 to 2009-07-01",
  "q2": "26.66%",
  "q3": "$6,063.41"
}

Selected options from execution:
Q1 -> 2009-06-01 to 2009-07-01
Q2 -> 26.66%
Q3 -> $6,063.41
